In [2]:
# Upload your ZIP file
from google.colab import files
uploaded = files.upload()

import zipfile, os, glob

# Get uploaded filename
zip_path = list(uploaded.keys())[0]
extract_dir = "/content/dataset"

# Extract all files
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_dir)

# List extracted CSVs
csv_files = glob.glob(os.path.join(extract_dir, "**/*.csv"), recursive=True)
print("Found CSV files:", len(csv_files))
for f in csv_files:
    print("-", os.path.relpath(f, extract_dir))


Saving archive (1).zip to archive (1).zip
Found CSV files: 2
- detect_dataset.csv
- classData.csv


In [3]:
import pandas as pd

dfs = []
for p in csv_files:
    df_i = pd.read_csv(p)
    df_i["_source_file"] = os.path.basename(p)
    dfs.append(df_i)

df = pd.concat(dfs, ignore_index=True, sort=False)

# Clean column names
df.columns = [c.strip() for c in df.columns]

print("Combined shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()


Combined shape: (19862, 14)
Columns: ['Output (S)', 'Ia', 'Ib', 'Ic', 'Va', 'Vb', 'Vc', 'Unnamed: 7', 'Unnamed: 8', '_source_file', 'G', 'C', 'B', 'A']


,Output (S),Ia,Ib,Ic,Va,Vb,Vc,Unnamed: 7,Unnamed: 8,_source_file,G,C,B,A
0,0.0,-170.472196,9.219613,161.252583,0.054490,-0.659921,0.605431,NaN,NaN,detect_dataset.csv,NaN,NaN,NaN,NaN
1,0.0,-122.235754,6.168667,116.067087,0.102000,-0.628612,0.526202,NaN,NaN,detect_dataset.csv,NaN,NaN,NaN,NaN
2,0.0,-90.161474,3.813632,86.347841,0.141026,-0.605277,0.464251,NaN,NaN,detect_dataset.csv,NaN,NaN,NaN,NaN
3,0.0,-79.904916,2.398803,77.506112,0.156272,-0.602235,0.445963,NaN,NaN,detect_dataset.csv,NaN,NaN,NaN,NaN
4,0.0,-63.885255,0.590667,63.294587,0.180451,-0.591501,0.411050,NaN,NaN,detect_dataset.csv,NaN,NaN,NaN,NaN


In [4]:
feature_cols = ['Va','Vb','Vc','Ia','Ib','Ic']   # adjust if needed
label_col = 'Output (S)'                         # your label column

df = df[feature_cols + [label_col]].dropna()
df[label_col] = pd.to_numeric(df[label_col], errors='coerce').astype(int)

X_base = df[feature_cols].values
y = df[label_col].values


In [5]:
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(sss.split(X_base, y))
X_train_base, X_test_base = X_base[train_idx], X_base[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

# Feature engineering
def add_features(X):
    Va,Vb,Vc,Ia,Ib,Ic = [X[:,i] for i in range(6)]
    LVab, LVbc, LVac = Va - Vb, Vb - Vc, Va - Vc
    Vm = np.sqrt(Va**2 + Vb**2 + Vc**2)
    Im = np.sqrt(Ia**2 + Ib**2 + Ic**2)
    I_imb = np.std(np.vstack([Ia,Ib,Ic]), axis=0)
    V_imb = np.std(np.vstack([Va,Vb,Vc]), axis=0)
    return np.column_stack([Va,Vb,Vc,Ia,Ib,Ic,LVab,LVbc,LVac,Vm,Im,I_imb,V_imb])

X_train_fe = add_features(X_train_base)
X_test_fe  = add_features(X_test_base)

scaler_tab = StandardScaler()
X_train_sc = scaler_tab.fit_transform(X_train_fe)
X_test_sc  = scaler_tab.transform(X_test_fe)


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

lr  = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
svm = SVC(kernel='rbf', C=10, gamma='scale', class_weight='balanced', probability=True, random_state=42)
dt  = DecisionTreeClassifier(class_weight='balanced', random_state=42)

lr.fit(X_train_sc, y_train)
svm.fit(X_train_sc, y_train)
dt.fit(X_train_sc, y_train)


DecisionTreeClassifier(class_weight='balanced', random_state=42)

In [7]:
SEQ_LEN = 50
X_seq = np.repeat(X_base[:, None, :], SEQ_LEN, axis=1)
X_seq += np.random.normal(0, 0.01, X_seq.shape)

X_train_seq, X_test_seq = X_seq[train_idx], X_seq[test_idx]

scaler_seq = StandardScaler()
N_tr, T, F = X_train_seq.shape
X_train_seq_sc = scaler_seq.fit_transform(X_train_seq.reshape(-1, F)).reshape(N_tr, T, F)
X_test_seq_sc  = scaler_seq.transform(X_test_seq.reshape(-1, F)).reshape(X_test_seq.shape[0], T, F)


In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class CNN_LSTM(nn.Module):
    def __init__(self, in_ch=6, num_classes=2):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(in_ch, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64), nn.ReLU(),
            nn.Conv1d(64,128,3,padding=1),
            nn.BatchNorm1d(128), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(128,256,3,padding=1),
            nn.BatchNorm1d(256), nn.ReLU(),
            nn.MaxPool1d(2)
        )
        self.lstm = nn.LSTM(input_size=256, hidden_size=128, batch_first=True, bidirectional=True)
        self.cls = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(128*2, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        x = x.permute(0,2,1)
        x = self.cnn(x)
        x = x.permute(0,2,1)
        h, _ = self.lstm(x)
        h = h.mean(dim=1)
        return self.cls(h)

model = CNN_LSTM(in_ch=X_train_seq_sc.shape[2], num_classes=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

train_ds = TensorDataset(torch.tensor(X_train_seq_sc, dtype=torch.float32),
                         torch.tensor(y_train, dtype=torch.long))
test_ds  = TensorDataset(torch.tensor(X_test_seq_sc, dtype=torch.float32),
                         torch.tensor(y_test, dtype=torch.long))

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=128)

# Train briefly
for epoch in range(5):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1} done")


Epoch 1 done
Epoch 2 done
Epoch 3 done
Epoch 4 done
Epoch 5 done


In [9]:
class TorchEstimator:
    def __init__(self, model, device):
        self.model = model
        self.device = device
        self.model.eval()
    def predict_proba(self, X):
        X_t = torch.tensor(X, dtype=torch.float32).to(self.device)
        with torch.no_grad():
            out = self.model(X_t)
            return torch.softmax(out, dim=1).cpu().numpy()

cnn = TorchEstimator(model, device)
probs_cnn = cnn.predict_proba(X_test_seq_sc)


In [10]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import math

def ensemble_and_eval(probs_ml, probs_cnn, y_test, name):
    # Weighted Ensemble
    weights = np.array([1, 2])  # 1 for ML model, 2 for CNN-LSTM
    stack_probs = (weights[0] * probs_ml + weights[1] * probs_cnn) / weights.sum()

    # Get hard predictions (0 or 1)
    y_pred = stack_probs.argmax(axis=1)

    print(f"\n{'='*20}")
    print(f"Results for: {name} + CNN-LSTM")
    print(f"{'='*20}")

    # --- Classification Metrics ---
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"F1 Macro: {f1_score(y_test, y_pred, average='macro'):.4f}")

    # --- Regression Metrics (Requested) ---
    # RMSE: Root Mean Squared Error
    mse = mean_squared_error(y_test, y_pred)
    rmse = math.sqrt(mse)

    # MAE: Mean Absolute Error
    mae = mean_absolute_error(y_test, y_pred)

    # R-squared
    r2 = r2_score(y_test, y_pred)

    print(f"RMSE:     {rmse:.4f}")
    print(f"MAE:      {mae:.4f}")
    print(f"R-Square: {r2:.4f}")

    # --- Detailed Report ---
    print("-" * 20)
    print("Classification Report:")
    print(classification_report(y_test, y_pred, digits=4))

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

# 1. Logistic Regression + CNN
ensemble_and_eval(lr.predict_proba(X_test_sc), probs_cnn, y_test, "Logistic Regression")

# 2. SVM + CNN
ensemble_and_eval(svm.predict_proba(X_test_sc), probs_cnn, y_test, "SVM")

# 3. Decision Tree + CNN
ensemble_and_eval(dt.predict_proba(X_test_sc), probs_cnn, y_test, "Decision Tree")



Results for: Logistic Regression + CNN-LSTM
Accuracy: 0.9875
F1 Macro: 0.9874
RMSE:     0.1118
MAE:      0.0125
R-Square: 0.9497
--------------------
Classification Report:
              precision    recall  f1-score   support

           0     0.9782    0.9992    0.9886      1301
           1     0.9991    0.9736    0.9862      1100

    accuracy                         0.9875      2401
   macro avg     0.9886    0.9864    0.9874      2401
weighted avg     0.9877    0.9875    0.9875      2401

Confusion Matrix:
[[1300    1]
 [  29 1071]]

Results for: SVM + CNN-LSTM
Accuracy: 0.9892
F1 Macro: 0.9891
RMSE:     0.1041
MAE:      0.0108
R-Square: 0.9564
--------------------
Classification Report:
              precision    recall  f1-score   support

           0     0.9826    0.9977    0.9901      1301
           1     0.9972    0.9791    0.9881      1100

    accuracy                         0.9892      2401
   macro avg     0.9899    0.9884    0.9891      2401
weighted avg     0.9893 